# Preparing Data

In [1]:
%pip install insightface onnxruntime opencv-python-headless pillow "numpy<2" pandas tqdm open_clip_torch torch --quiet

Note: you may need to restart the kernel to use updated packages.


##### Loading models

In [2]:
import gc
import pickle
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import open_clip
from PIL import Image
from tqdm.auto import tqdm
from insightface.app import FaceAnalysis

# ArcFace
print("Loading ArcFace (buffalo_s)...")
app = FaceAnalysis(name="buffalo_s", providers=["CPUExecutionProvider"])
app.prepare(ctx_id=0, det_size=(320, 320))

# CLIP
print("Loading CLIP (ViT-B/32)...")
clip_device = "cpu"
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k"
)
clip_model.eval().to(clip_device)

print("Both models ready.")

/Users/brandley/.pyenv/versions/3.11.3/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading ArcFace (buffalo_s)...
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/brandley/.insightface/models/buffalo_s/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/brandley/.insightface/models/buffalo_s/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/brandley/.insightface/models/buffalo_s/det_500m.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/brandley/.insightface/models/buffalo_s/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/brandley/.insightface/models/buffalo_s/w600k_mbf.onnx r

##### Path and parameter info for ArcFace and CLIP models

In [9]:
# Folder structure
IMAGES_DIR        = Path("images")                        # input: images/<team>/<player>/*.jpg

# Output files
ARC_FILE          = Path("player_embeddings.npz")         # ArcFace per-player means
ARC_PER_IMAGE     = Path("per_image_embeddings.npz")      # ArcFace per-image vectors
CLIP_FILE         = Path("clip_player_embeddings.npz")    # CLIP per-player means
KEPT_MAP_FILE     = Path("kept_map.pkl")                  # filenames that passed outlier filter
CHECKPOINT_FILE   = Path("per_image_checkpoint.pkl")      # crash-resume for ArcFace walk

# ArcFace thresholds
MIN_FACE_SIZE     = 80      # reject faces smaller than this
MAX_IMAGE_SIDE    = 1024    # downscale anything larger before inference
OUTLIER_THRESH    = 0.30    # drop photos with cosine similarity below this vs the provisional mean
MIN_KEEP          = 3       # never drop a player below this many photos
GC_EVERY          = 50
CHECKPOINT_EVERY  = 200

# To force a fresh rebuild rather than resume from checkpoint:
# CHECKPOINT_FILE.unlink(missing_ok=True)

In [10]:
def biggest_face_embedding(img_path):
    """Return (embedding, bbox) for the largest detected face, or (None, None)."""
    try:
        with Image.open(img_path) as pil:
            pil = pil.convert("RGB")
            if max(pil.size) > MAX_IMAGE_SIDE:
                pil.thumbnail((MAX_IMAGE_SIDE, MAX_IMAGE_SIDE))
            img = np.array(pil)
    except Exception:
        return None, None
    try:
        faces = app.get(img)
    except Exception:
        return None, None
    if not faces:
        return None, None
    faces.sort(key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]),
               reverse=True)
    f = faces[0]
    w, h = f.bbox[2] - f.bbox[0], f.bbox[3] - f.bbox[1]
    if min(w, h) < MIN_FACE_SIZE:
        return None, None
    return f.normed_embedding, f.bbox


# Walk images/<team>/<player>/*.jpg
all_images, player_keys = [], set()
for team_dir in sorted(IMAGES_DIR.iterdir()):
    if not team_dir.is_dir():
        continue
    for player_dir in sorted(team_dir.iterdir()):
        if not player_dir.is_dir():
            continue
        key = f"{team_dir.name}__{player_dir.name}"
        player_keys.add(key)
        for img_path in sorted(player_dir.glob("*.jpg")):
            all_images.append((key, img_path))

# Resume from checkpoint if it exists
if CHECKPOINT_FILE.exists():
    with open(CHECKPOINT_FILE, "rb") as f:
        per_image, no_face, done_paths = pickle.load(f)
    print(f"Resuming: {sum(len(v) for v in per_image.values())} embeddings done, "
          f"{len(done_paths)} images processed.")
else:
    per_image, no_face, done_paths = {}, [], set()

print(f"Scanning {len(all_images)} images across {len(player_keys)} players...")
for i, (player, path) in enumerate(tqdm(all_images), 1):
    spath = str(path)
    if spath in done_paths:
        continue
    emb, _ = biggest_face_embedding(path)
    done_paths.add(spath)
    if emb is None:
        no_face.append((player, path.name))
    else:
        per_image.setdefault(player, []).append((path.name, emb))
    if i % GC_EVERY == 0:
        gc.collect()
    if i % CHECKPOINT_EVERY == 0:
        with open(CHECKPOINT_FILE, "wb") as f:
            pickle.dump((per_image, no_face, done_paths), f)

with open(CHECKPOINT_FILE, "wb") as f:
    pickle.dump((per_image, no_face, done_paths), f)

print(f"\nDetected faces in {sum(len(v) for v in per_image.values())} / {len(all_images)} images")
print(f"Players with zero detected faces: {len(player_keys - set(per_image.keys()))}")

Resuming: 17151 embeddings done, 23623 images processed.
Scanning 21980 images across 1271 players...


100%|██████████| 21980/21980 [00:00<00:00, 1505450.90it/s]


Detected faces in 17151 / 21980 images
Players with zero detected faces: 66


## Summary of kept and dropped pictures per player

In [11]:
clean    = {}
kept_map = {}

for player, items in per_image.items():
    if not items:
        continue
    files = [f for f, _ in items]
    embs  = np.stack([e for _, e in items])

    if len(embs) == 1:
        clean[player] = embs[0]
        kept_map[player] = files
        continue

    # Provisional mean
    mean = embs.mean(axis=0)
    mean /= np.linalg.norm(mean) + 1e-9
    sims = embs @ mean

    # Drop low scorers, but never below MIN_KEEP
    keep = sims >= OUTLIER_THRESH
    if keep.sum() < MIN_KEEP:
        idx = np.argsort(-sims)[:MIN_KEEP]
        keep = np.zeros(len(embs), dtype=bool)
        keep[idx] = True

    # Final mean over the kept set
    kept = embs[keep]
    m = kept.mean(axis=0)
    m /= np.linalg.norm(m) + 1e-9
    clean[player]    = m
    kept_map[player] = [f for f, k in zip(files, keep) if k]

# Diagnostic summary
rows = [{"player":  p,
         "kept":    len(kept_map[p]),
         "dropped": len(per_image[p]) - len(kept_map[p]),
         "total":   len(per_image[p])}
        for p in per_image if per_image[p]]
summary = pd.DataFrame(rows).sort_values("dropped", ascending=False)
print(f"Built {len(clean)} player fingerprints.")
print("\nTop-10 players with most rejected photos:")
print(summary.head(10).to_string(index=False))

Built 1205 player fingerprints.

Top-10 players with most rejected photos:
                       player  kept  dropped  total
      Norway__Marcus Pedersen    12        2     14
         Ghana__Gideon Mensah    12        1     13
    Uruguay__Luciano González    10        1     11
          Portugal__Rui Silva    10        1     11
          Egypt__Mohamed Alaa     5        1      6
      Senegal__Abdoulaye Seck     9        1     10
Uzbekistan__Behruzjon Karimov     7        1      8
     South Korea__Kim Ju-Sung    11        1     12
               Brazil__Ibañez     5        1      6
        Algeria__Achref Abada     6        0      6


In [12]:
np.savez(ARC_FILE,
         **{p: v.astype(np.float32) for p, v in clean.items()})
np.savez(ARC_PER_IMAGE,
         **{f"{p}__{f}": e.astype(np.float32)
            for p, items in per_image.items() for f, e in items})
with open(KEPT_MAP_FILE, "wb") as f:
    pickle.dump(kept_map, f)

print(f"Saved {len(clean)} player fingerprints -> {ARC_FILE}")
print(f"Saved per-image vectors            -> {ARC_PER_IMAGE}")
print(f"Saved kept_map                     -> {KEPT_MAP_FILE}")

Saved 1205 player fingerprints -> player_embeddings.npz
Saved per-image vectors            -> per_image_embeddings.npz
Saved kept_map                     -> kept_map.pkl


## CLIP model

### CLIP per-image embeddings

##### Loading players for CLIP embeddings

In [17]:
if not KEPT_MAP_FILE.exists():
    raise FileNotFoundError(
        f"Couldn't find {KEPT_MAP_FILE}. Run the ArcFace pipeline first "
        f"(or save kept_map there) so CLIP knows which images to embed."
    )

with open(KEPT_MAP_FILE, "rb") as f:
    kept_map = pickle.load(f)

total_imgs = sum(len(v) for v in kept_map.values())
print(f"Loaded kept_map: {len(kept_map)} players, {total_imgs} total kept images.")

Loaded kept_map: 1205 players, 17141 total kept images.


##### CLIP embedding function

In [18]:
@torch.no_grad()
def clip_embed(img_path):
    """Return the L2-normalized 512-d CLIP image embedding, or None on failure."""
    try:
        with Image.open(img_path) as pil:
            pil = pil.convert("RGB")
            tensor = clip_preprocess(pil).unsqueeze(0).to(clip_device)
        feat = clip_model.encode_image(tensor)
        feat = feat / feat.norm(dim=-1, keepdim=True)
        return feat.squeeze(0).cpu().numpy()
    except Exception:
        return None

##### Embed all images, with checkpoint

In [24]:
# Build the work queue from kept_map
work = []
for player_key, kept_files in kept_map.items():
    team, player = player_key.split("__", 1)
    folder = IMAGES_DIR / team / player
    for f in kept_files:
        p = folder / f
        if p.exists():
            work.append((player_key, f, p))

# Resume from checkpoint if it exists
if CLIP_CHECKPOINT_FILE.exists():
    with open(CLIP_CHECKPOINT_FILE, "rb") as f:
        clip_per_image, clip_done_paths = pickle.load(f)
    print(f"Resuming: {sum(len(v) for v in clip_per_image.values())} "
          f"CLIP embeddings already done, "
          f"{len(clip_done_paths)} images processed.")
else:
    clip_per_image, clip_done_paths = {}, set()
    print("No checkpoint found — starting fresh.")

print(f"CLIP-embedding {len(work)} kept images across {len(kept_map)} players...")
for i, (player_key, filename, path) in enumerate(tqdm(work), 1):
    spath = str(path)
    if spath in clip_done_paths:
        continue   # already done in a previous run

    emb = clip_embed(path)
    clip_done_paths.add(spath)
    if emb is None:
        continue   # skip un-embeddable images silently

    clip_per_image.setdefault(player_key, []).append((filename, emb))

    # Periodic checkpoint
    if i % CHECKPOINT_EVERY == 0:
        with open(CLIP_CHECKPOINT_FILE, "wb") as f:
            pickle.dump((clip_per_image, clip_done_paths), f)

# Final checkpoint after the loop completes
with open(CLIP_CHECKPOINT_FILE, "wb") as f:
    pickle.dump((clip_per_image, clip_done_paths), f)

print(f"\nDone. CLIP embeddings for {len(clip_per_image)} players, "
      f"{sum(len(v) for v in clip_per_image.values())} total vectors.")

Resuming: 17141 CLIP embeddings already done, 17141 images processed.
CLIP-embedding 17141 kept images across 1205 players...


100%|██████████| 17141/17141 [00:00<00:00, 3482420.19it/s]


Done. CLIP embeddings for 1205 players, 17141 total vectors.


In [22]:
clip_clean = {}
for player_key, items in clip_per_image.items():
    embs = np.stack([e for _, e in items])
    m = embs.mean(axis=0)
    m /= np.linalg.norm(m) + 1e-9
    clip_clean[player_key] = m

print(f"Built {len(clip_clean)} CLIP fingerprints.")

Built 1205 CLIP fingerprints.


In [23]:
np.savez(CLIP_FILE,
         **{p: v.astype(np.float32) for p, v in clip_clean.items()})
print(f"Saved {len(clip_clean)} CLIP fingerprints -> {CLIP_FILE}")

Saved 1205 CLIP fingerprints -> clip_player_embeddings.npz
